# Dataset Real Estate Listings (Data.World)

## Creating Folders

In [ ]:
!echo "Creating data folder"
!mkdir -p data/bronze data/silver data/gold

# Downloading Dataset

In [ ]:
import pandas as pd
df = pd.read_csv('https://query.data.world/s/3ox2v5trynvowrhc27fjqkcx3lytjf?dws=00000')
df.to_csv("data/bronze/properati-BR-2016-11-01-properties-rent.csv", index=False)

In [ ]:
df.info()

In [ ]:
df.head()

## Cleaning (Silver)

## Split 'place' column

In [ ]:
def split_place_with_parent_names(src_df):
    dst_df = src_df.copy()
    tmp_df = src_df['place_with_parent_names'].str.split("|", expand=True)

    dst_df['country'] = tmp_df[1]
    dst_df['region'] = tmp_df[2]
    dst_df['city'] = tmp_df[3]

    return dst_df

df_split = split_place_with_parent_names(df)
df_split

## Remove unused columns

In [ ]:
def remove_unused_colums(src_def):
    return df.drop(columns=[
        'place_name',
        'geonames_id',
        'place_name',
        'place_with_parent_names',
        'lat-lon',
        'currency',
        'price_aprox_local_currency',
        'expenses',
        'properati_url',
        'description',
        'title',
        'image_thumbnail']
    )

df_less_columns = remove_unused_colums(df_split)
df_less_columns.info()

## Count # of NaN values

In [ ]:
df_less_columns.isna().sum()

In [ ]:
def replace_empty_values(src_df):
    dst_df = src_df.copy()
    # dst_df["geonames_id"] = src_df["geonames_id"].fillna(0)
    # dst_df["lat-lon"] = src_df["lat-lon"].fillna("")
    dst_df["lat"] = src_df["lat"].fillna(0)
    dst_df["lon"] = src_df["lon"].fillna(0)

    dst_df["price"] = src_df["price"].fillna(0)
    # dst_df["currency"] = src_df["currency"].fillna("")
    
    # dst_df["price_aprox_local_currency"] = src_df["price_aprox_local_currency"].fillna(0)
    dst_df["price_aprox_usd"] = src_df["price_aprox_usd"].fillna(0)
    dst_df["surface_total_in_m2"] = src_df["surface_total_in_m2"].fillna(0)
    dst_df["surface_covered_in_m2"] = src_df["surface_covered_in_m2"].fillna(0)
    dst_df["price_usd_per_m2"] = src_df["price_usd_per_m2"].fillna(0)
    dst_df["price_per_m2"] = src_df["price_per_m2"].fillna(0)

    dst_df["floor"] = src_df["floor"].fillna(0)
    dst_df["rooms"] = src_df["rooms"].fillna(0)

    # dst_df["expenses"] = src_df["expenses"].fillna(0)
    # dst_df["image_thumbnail"] = src_df["image_thumbnail"].fillna("")


    return dst_df

df_filled = replace_empty_values(df_less_columns)
df_filled.isna().sum()

## Drop entries without price

In [ ]:
def drop_entries_without_price(src_df):
    return src_df[src_df["price"] != 0].copy()

df_priced = drop_entries_without_price(df_filled)
df_priced

## Update USD values

In [ ]:
USD_TO_BRL = 5.06 # 1 USD = X BRL

def update_usd_values(src_df):
    dst_df = src_df.copy()

    usd_price = src_df['price'] / USD_TO_BRL
    dst_df['price_aprox_usd'] = usd_price

    valid_surface = src_df['surface_covered_in_m2'] > 0
    dst_df.loc[valid_surface, 'price_usd_per_m2'] = usd_price / src_df['surface_covered_in_m2']

    return dst_df

df_usd = update_usd_values(df_priced)
df_usd

In [ ]:
df_usd.info()

In [ ]:
df_usd.to_csv('data/silver/properati-BR-2016-11-01-properties-rent.csv', index=False)